# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](grid.png)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [4]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        S = []
        for r in range(self.height):
            for c in range(self.width):
                if self.is_valid_state((r, c)):
                    S.append((r, c))
        return S

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            success_prob = 0.6
            fail_prob = 0.2
        else:
            success_prob = 0.9
            fail_prob = 0.05

        if action in [(-1, 0), (1, 0)]:
            deviations = [(0, -1), (0, 1)]
        else:
            deviations = [(-1, 0), (1, 0)]
            
        def move(s, act):
            r, c = s
            ar, ac = act
            ns = (r + ar, c + ac)
            return ns if self.is_valid_state(ns) else s
            
        outcomes = {}
        
        # Forward
        n_s = move(state, action)
        outcomes[n_s] = outcomes.get(n_s, 0) + success_prob
        
        # Deviations
        for dev in deviations:
            n_s = move(state, dev)
            outcomes[n_s] = outcomes.get(n_s, 0) + fail_prob
            
        return [(s_prime, p) for s_prime, p in outcomes.items()]




### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [5]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [6]:
def expected_next_value(grid, state, action, V):
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )
    # sum_{s'} T(s,a,s') V(s')


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}
    deltas = []
    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0


        # Aplicamos la ecuación de Bellman a cada estado s.
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )

                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
            # Cambio local del estado:
            # |V_{k+1}(s) - V_k(s)|
                abs(V_new[state] - V[state])
            )


        # Terminamos la iteración:
        # V_{k+1} pasa a ser V_k para la siguiente vuelta.
        V = V_new
        deltas.append(biggest_change)


        
        if biggest_change < threshold:
            break

    return V, iteration + 1, deltas


def extract_policy(grid, V):
    policy = {}


   
    for state in grid.states():
        if grid.is_terminal(state):
            continue

        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return policy

